# Practice : Open-Source sLLM 실행 및 비교

[Task]
1. Hugging Face에서 sLLM을 직접 다운로드하여 실행
2. 동일 프롬프트에 대한 범용 LLM(ChatGPT, Claude 등) / Base 모델 / Instruct 모델의 응답 비교
3. 특정 업무(Text-to-SQL)에 특화 튜닝된 모델의 출력 확인

[Model]
- Base / Instruct 비교: Qwen2.5-1.5B, Qwen2.5-1.5B-Instruct
- 특화 튜닝 확인: SLM-SQL-0.5B

[Note]
- Colab 환경을 기본으로 작성, 로컬 환경 실행 시 안내를 별도 표기
- 모두 공개(gated 아님) 모델이므로 별도 로그인 불필요

## 1. Colab GPU 환경 설정 [Colab 전용]

- 런타임 > 런타임 유형 변경 > 하드웨어 가속기: T4 GPU 선택 후 저장
- 로컬 환경 사용자는 이 섹션 건너뛰기
- 무료 Colab은 세션 시간과 GPU 사용량에 제한이 있음. 연결이 끊기면 런타임을 다시 시작하고 위 셀부터 재실행

In [ ]:
!nvidia-smi

In [3]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU 사용 가능: False
Device: CPU


## 2. 라이브러리 설치

- transformers 최신 버전 필요 (구버전 사용 시 모델 설정 인식 오류 발생 가능)
- 로컬 환경에서 이미 설치되어 있다면 생략 가능

In [1]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 72.8 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.14.1
    Uninstalling transformers-5.14.1:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-5.14.1━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


## 3. 라이브러리 Import

In [2]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)

# GPU마다 지원하는 연산 정밀도가 다름 (예: T4는 bfloat16 미지원, fp16만 가속됨)
# 사용 가능한 최적 dtype을 자동 선택
if torch.cuda.is_available():
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    compute_dtype = torch.float32

print("Transformers version:", transformers.__version__)
print("Compute dtype:", compute_dtype)

NameError: name 'torch' is not defined

[Note] 모델 다운로드 저장 위치
- Hugging Face 캐시 경로"~/.cache/huggingface/hub"에 저장됨
- Colab은 런타임(가상 인스턴스)의 로컬 디스크에 저장 : 런타임 종료 시 캐시도 함께 삭제되어 재접속 시 재다운로드 필요
- 로컬 환경은 컴퓨터에 캐시가 계속 남아있어 재실행 시 다운로드 없이 바로 로드됨

## 4. Base 모델 로드 (Qwen2.5-1.5B)

- Instruction Tuning이 적용되지 않은 사전학습 모델
- Chat Template 없이 순수 텍스트 이어쓰기 방식으로 동작

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=compute_dtype,
    device_map="auto"
)

print("Base 모델 로드 완료:", base_model_id)

## 5. Instruct 모델 로드 (Qwen2.5-1.5B-Instruct)

- Instruction Tuning이 적용된 모델
- Chat Template을 적용하여 대화 형식으로 입력

In [ ]:
instruct_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_id)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_id,
    torch_dtype=compute_dtype,
    device_map="auto"
)

print("Instruct 모델 로드 완료:", instruct_model_id)

## 6. 추론 함수 정의

- Base: 프롬프트를 그대로 이어쓰기
- Instruct: Chat Template 적용 후 생성

[Note] Chat Template ?
- Instruct 모델은 "<|im_start|>user ... <|im_end|>"처럼 발화자를 구분하는 특수 토큰 구조로 학습됨
- apply_chat_template()은 이 구조를 자동으로 만들어주는 함수이며, Base 모델은 이런 구조를 학습한 적이 없어 사용하지 않음

In [ ]:
def generate_base(prompt: str, max_new_tokens: int = 200) -> str:
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    output_ids = base_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return base_tokenizer.decode(generated, skip_special_tokens=True)


def generate_instruct(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    # return_dict=True: transformers 버전에 따라 apply_chat_template이
    # 텐서 대신 BatchEncoding을 반환하는 경우가 있어 명시적으로 지정
    inputs = instruct_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(instruct_model.device)
    output_ids = instruct_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return instruct_tokenizer.decode(generated, skip_special_tokens=True)

## 7. 비교 프롬프트 세트

[Note]
- 범용 LLM(ChatGPT, Claude 등) 결과는 아래 프롬프트를 웹 UI에 직접 질의하여 별도로 기록
- 이 노트북에서는 sLLM(Base/Instruct)만 코드로 직접 비교

In [ ]:
single_turn_prompts = {
    "1_사실질문": "세종대왕이 만든 문자는 무엇인가요?",
    "2_지시문형": (
        "다음 문장을 한 문장으로 요약해줘: "
        "'인공지능 기술의 발전으로 다양한 산업 분야에서 자동화가 가속화되고 있으며, "
        "특히 제조업과 금융업에서 그 변화가 두드러지게 나타나고 있다.'"
    ),
    "3_추론문제": (
        "한 반에 학생이 30명 있습니다. 이 중 안경을 쓴 학생은 12명이고, "
        "안경을 쓴 학생 중 여학생은 7명입니다. "
        "안경을 쓰지 않은 학생 중 남학생이 10명이라면, "
        "안경을 쓰지 않은 여학생은 몇 명인가요?"
    ),
}

for name, prompt in single_turn_prompts.items():
    print(f"[{name}] {prompt}\n")

## 8. 단일 턴 프롬프트 실행 및 비교

[Note]
- Base 모델은 종료 시점을 스스로 판단하지 못해 매번 max_new_tokens만큼 끝까지 생성 : Instruct보다 시간이 더 걸림


In [ ]:
# 실행 전 device 확인: cpu로 나오면 생성이 매우 느릴 수 있음
print("Base 모델 device:", base_model.device, "| dtype:", base_model.dtype)
print("Instruct 모델 device:", instruct_model.device, "| dtype:", instruct_model.dtype)

if base_model.device.type == "cpu":
    print("\n GPU가 아닌 CPU에서 실행 중")


In [ ]:
for name, prompt in single_turn_prompts.items():
    print(f"=== {name} ===")
    print("[Prompt]", prompt)
    print("\n[Base]")
    print(generate_base(prompt))
    print("\n[Instruct]")
    print(generate_instruct(prompt))
    print("\n" + "=" * 60 + "\n")

## 9. 멀티턴 대화 비교

[Note]
- Base 모델은 대화 형식(Chat Template)을 학습하지 않았으므로, 직전 턴과 답변을 프롬프트에 이어붙이는 방식으로 실행
- Instruct 모델은 messages 리스트에 대화 히스토리를 누적하여 실행

In [ ]:
turn1 = "제주도 여행 코스를 하나 추천해줘"
turn2 = "방금 그거, 1박 2일로 압축해줄 수 있어?"

# Base: 이전 턴과 응답을 프롬프트에 이어붙임
base_turn1_response = generate_base(turn1)
base_multiturn_prompt = f"{turn1}\n{base_turn1_response}\n{turn2}"
base_turn2_response = generate_base(base_multiturn_prompt)

print("[Base - Turn1]", base_turn1_response)
print("\n[Base - Turn2]", base_turn2_response)

In [ ]:
# Instruct: messages 리스트에 대화 히스토리 누적
messages = [{"role": "user", "content": turn1}]
inputs = instruct_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(instruct_model.device)
output_ids = instruct_model.generate(**inputs, max_new_tokens=400, do_sample=False)
instruct_turn1_response = instruct_tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)

messages.append({"role": "assistant", "content": instruct_turn1_response})
messages.append({"role": "user", "content": turn2})

inputs = instruct_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(instruct_model.device)
output_ids = instruct_model.generate(**inputs, max_new_tokens=400, do_sample=False)
instruct_turn2_response = instruct_tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)

print("[Instruct - Turn1]", instruct_turn1_response)
print("\n[Instruct - Turn2]", instruct_turn2_response)

## 10. 범용 LLM과의 비교 기록

[Task]
- 위 프롬프트(1~4)를 ChatGPT, Claude 등 범용 LLM 웹 UI에 동일하게 질의
- 아래 표에 결과를 요약하여 정리

| 프롬프트 | 범용 LLM | sLLM-Base | sLLM-Instruct |
|---|---|---|---|
| 1. 사실 질문 | | | |
| 2. 지시문형 | | | |
| 3. 추론 문제 | | | |
| 4. 멀티턴 | | | |

# 특화 튜닝 모델 확인 (Text-to-SQL)

## 11. 개요

[Task]
- 특정 업무(Text-to-SQL)에 특화 튜닝된 모델의 응답을 직접 실행하여 확인
- baseline과의 실행 비교 없이, 특화 모델 하나의 출력만 확인

[Model]
- 특화 튜닝: SLM-SQL-0.5B (Qwen2.5-Coder-0.5B-Instruct 기반, SFT + RL 튜닝)

[Note: Model Description]

 - Qwen2.5-Coder-0.5B-Instruct(범용 코드 모델)를 Text-to-SQL 작업에 SFT + RL로 추가 튜닝한 모델
- BIRD 벤치마크 기준, 같은 0.5B 파라미터에서도 튜닝 전후 정확도 차이가 크게 발생

| 모델 상태 | Execution Accuracy |
|---|---|
| 튜닝 전 (일반 SFT만, Qwen2.5-Coder-0.5B-Instruct) | 42.13% |
| 튜닝 후 (SFT + RL, SLM-SQL-0.5B) | 65.31% |

- (출처: SLM-SQL 논문, arXiv:2507.22478)


## 12. SQL 모델 로드

In [ ]:
sql_tuned_id = "cycloneboy/SLM-SQL-0.5B"

sql_tuned_tokenizer = AutoTokenizer.from_pretrained(sql_tuned_id)
sql_tuned_model = AutoModelForCausalLM.from_pretrained(
    sql_tuned_id, torch_dtype=compute_dtype, device_map="auto"
)

print("특화 SQL 모델 로드 완료:", sql_tuned_id)

## 13. SQL 생성 프롬프트 정의

- 스키마 정보를 함께 제공하고, 자연어 질문을 SQL로 변환

In [ ]:
sql_schema = """
employees(id, name, dept_id, salary)
departments(dept_id, dept_name, location)
"""
sql_question = "서울에 위치한 부서별 평균 급여를 높은 순으로 보여줘."

sql_prompt = f"""다음 스키마를 참고하여 질문에 맞는 SQL 쿼리만 작성하세요.

스키마: {sql_schema}
질문: {sql_question}
SQL:"""

print(sql_prompt)

## 14. SQL 생성 실행

In [ ]:
def generate_sql(tokenizer, model, prompt: str, max_new_tokens: int = 150) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False
    )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


tuned_sql = generate_sql(sql_tuned_tokenizer, sql_tuned_model, sql_prompt)

print("=== 특화 튜닝 모델: SLM-SQL-0.5B ===")
print(tuned_sql)

# [로컬 전용 실습] 네트워크 연결 없이 실행하기

## 15. 오프라인 추론 확인

[Task]
- 위 셀들을 모두 실행하여 모델 다운로드가 완전히 끝난 것을 확인한 뒤 진행
- Wi-Fi 또는 랜선을 분리한 상태에서 아래 셀을 실행하여, 로컬에 저장된 sLLM이 인터넷 없이도 동작하는지 확인

[Note]
- Colab은 클라우드 가상 환경이라 이 실습을 수행할 수 없음 (로컬 환경 전용)
- Colab에서는 (네트워크가 연결되어 있다면) "캐시 재사용이 잘 되는지"만 확인됨
- local_files_only=True 옵션으로 캐시된 파일만 사용하도록 강제

In [ ]:
# 네트워크를 분리한 뒤 이 셀을 실행
offline_tokenizer = AutoTokenizer.from_pretrained(
    instruct_model_id, local_files_only=True
)
offline_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_id,
    torch_dtype=compute_dtype,
    device_map="auto",
    local_files_only=True
)

messages = [{"role": "user", "content": "sLLM이 무엇인지 한 문장으로 설명해줘"}]
inputs = offline_tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(offline_model.device)
output_ids = offline_model.generate(**inputs, max_new_tokens=100, do_sample=False)

print(offline_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))